In [ ]:
from pathlib import Path

import pandas as pd
import torch
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PROJECT_ROOT = Path.cwd().parent  # If notebook is in notebooks/
DATA_DIR = PROJECT_ROOT / "data"
df = pd.read_csv(DATA_DIR / "final_data.csv")

import torch

model = torch.load("../models/mel-vs-nv-full.pth", map_location=device, weights_only=False)
model.eval()
model = model.to(device)

In [ ]:
x = torch.randn(1, 3, 224, 224, device=device)

output = model(x)

# print(output.shape)
print(output)

In [ ]:
target_layers = [model.get_submodule(f"folds.{fold_name}.encoder.layer4.1.conv2") for fold_name, _ in model.folds.named_children()]

print(target_layers)

In [ ]:
# create the Grad CAM Wrapper
from torch import nn


class GradCAMWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        return self.model(x)[0].reshape(-1, 1)


cam_model = GradCAMWrapper(model)

In [ ]:
# from pytorch_grad_cam import GradCAM
# from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# cam = GradCAM(
#     model=cam_model,
#     target_layers=[target_layer]
# )

# targets = [ClassifierOutputTarget(0)]

In [ ]:
from pathlib import Path

import numpy as np
import torch
from PIL import Image

image_path = Path("../data") / df.iloc[0]["path"]

image = Image.open(image_path).convert("RGB")

In [ ]:
print(type(image))
print(image.size)

In [ ]:
from torchvision import transforms

input_tensor = transforms.PILToTensor()(image).unsqueeze(0)
input_tensor = input_tensor.to(device=device, dtype=torch.float32).requires_grad_(True)
print(input_tensor.shape)

In [ ]:
cam = GradCAM(model=cam_model, target_layers=target_layers)

In [ ]:
called_layers = []


def record_layer(module, inputs, output):
    called_layers.append(module)


handles = []

for name, module in model.named_modules():
    if name.startswith("folds.0.encoder.layer4"):
        handles.append(module.register_forward_hook(lambda m, i, o, name=name: called_layers.append(name)))

with torch.no_grad():
    model(input_tensor)

for handle in handles:
    handle.remove()

print(called_layers)

In [ ]:
# Grad-CAM accepts ClassifierOutputTarget as a callable.
grayscale_cam = cam(input_tensor=input_tensor, targets=[ClassifierOutputTarget(0)])  # pyright: ignore[reportArgumentType]

grayscale_cam = grayscale_cam[0]

In [ ]:
import matplotlib.pyplot as plt

# Original image as RGB, scaled to [0, 1]
rgb_img = np.array(image).astype(np.float32) / 255.0

visualization = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

plt.figure(figsize=(10, 7))
plt.imshow(visualization)
plt.axis("off");